# 用合成数据学习 CST／CRP 追踪审核

这里用可控的合成 b0、标签和流线说明每层审核。正式使用 [分步处理入口](Tractography.ipynb)，并结合实际解剖和图谱版本判断。先按顺序运行各格，再逐项尝试控件。 真实公开 DWI 请按 [HCP Young Adult 接入指南](../docs/PUBLIC_DATA.md#hcp-young-adult)准备校正后 DWI、同一系列梯度和已审核变换。

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import tempfile

cwd = Path.cwd()
repo = next((p for p in (cwd, *cwd.parents) if (p / "run_tractography.py").is_file()), None)
if repo is None:
    raise RuntimeError("请从 NEO_MRI 仓库或其 notebooks 目录打开。")
sys.path.insert(0, str(repo))
sys.path.insert(0, str(repo / "code"))

from IPython.display import FileLink, display
from tractography.viewer import interactive

%matplotlib inline

def cli(stage, output, **options):
    # 使用当前内核的 Python，并直接调用同一套命令行后端。
    argv = [sys.executable, "-B", str(repo / "run_tractography.py"), stage, "--output", str(output)]
    for name, value in options.items():
        argv += [f"--{name}", str(value)]
    result = subprocess.run(argv, cwd=repo, capture_output=True, encoding="utf-8",
                            env={**os.environ, "PYTHONIOENCODING": "utf-8", "MPLBACKEND": "Agg"})
    if result.returncode:
        raise RuntimeError(result.stderr or result.stdout)
    return json.loads(result.stdout)

def link(path):
    path = Path(path).resolve()
    print(path)
    if path.is_relative_to(repo):
        display(FileLink(Path("..") / path.relative_to(repo) if cwd == repo / "notebooks"
                         else path.relative_to(repo)))

# 演示输出留在检出目录下的 work，每次执行生成新的目录。
work = repo / "work"
work.mkdir(exist_ok=True)

## 1. 看懂三个数

播种尝试数是启动追踪的预算。保存数是父 TCK 中的流线数。候选数是从该父集通过指定解剖门的流线数。比较两个门时先确认它们用同一父集。

In [ ]:
output = Path(tempfile.mkdtemp(prefix="tractography_tutorial_", dir=work))
example = cli("demo", output / "synthetic")
results = Path(example["results"])
manifest = json.loads((results / "results.json").read_text(encoding="utf-8"))
link(example["report"])
run = next(row for row in manifest["runs"] if row["id"] == "ST_iFOD2_L_M1")
for name, count in (("播种尝试", run["attempts"]), ("保存流线", run["saved"]),
                    ("符合向下条件", run["metrics"]["descending"])):
    print(f"{name} {count}")

## 2. 改一个解剖门，保持同一个父集

严格门要求同侧流线依次经过内囊后肢、脑脚和原始脑桥门。扩展门把内囊后肢门在可追踪脑域内向外扩 2 mm。对比新增流线实际从哪里通过，留意邻近白质被纳入的可能性。

Navigator_mRt 筛查经过中脑网状结构门的候选，Navigator_sMRt 与 Navigator_GRN 还要求按顺序经过各自的下游门。图谱定义、源空间和映射质量都会影响候选数量。

In [ ]:
strict = "CST_PLIC_CP_Tang_pons"
expanded = "CST_PLIC_plus2_CP_Tang_pons"
for gate in (strict, expanded, "Tang_pons_medulla", "Navigator_mRt",
             "Navigator_sMRt", "Navigator_GRN"):
    print(gate, run["gates"][gate]["selected"])

## 3. 看覆盖图与核心图

切换「原始父流线」与「筛选后流线」，观察门筛出了哪一段路径。覆盖计数中每条流线在每个体素最多计一次。核心图对每个原生物理高度层保留达到该层峰值 15% 的已访问体素。切换图层后核对候选数保持相同。

再切到另一个模型或侧别。选 M2 时可以在同一侧分别查看 SMA 和 preSMA。Tensor_Det 菜单只呈现该方案定义的门。

In [ ]:
controls = interactive(results)
controls["model"].value = "ST_iFOD2"
controls["origin"].value = "M1"
controls["side"].value = "L"
controls["gate"].value = expanded
controls["view"].value = "逐层 15% 核心"
controls["show"].click()

## 4. 记录该复查的技术分支

播种阈值决定可从哪里起跑，传播阈值决定已启动的流线何时停止。解剖门在追踪结束后筛选已存在的流线。正式数据中某个门没有候选时，回看 b0 脑干覆盖、图谱定位、追踪支持区、模型与流线数，再解释其意义。

审核时记录同一分母下的计数、各标签在原生空间的位置和代表性切片。合成数据的计数用于理解规则。个体研究中的病灶负荷和功能判断需要另外的证据。